# SQL Refresh: Exploring the IMDb Database

**Course:** Database Applications Development  
**Medina County Career Center**

---

Welcome back to SQL! We've been away from databases for a bit, so today we're going to dust off our SQL skills with a brand new database — **IMDb movie data**.

This database has ~19,000 movies and TV series from 2010 to present, along with ratings, actors, directors, and crew. It's all titles you'll actually recognize.

**Today's goals:**
1. Reconnect to a database and remember how it works
2. Explore the structure of a new database
3. Practice SELECT, WHERE, ORDER BY, and GROUP BY
4. Find some movies and actors YOU care about

## Setup

First, let's import our libraries and connect to the database. Same pattern as before — nothing new here.

In [24]:
import pandas as pd
import sqlite3

# Connect to the filtered IMDb database
dbPath = 'imdb_class_2010plus.db'
connection = sqlite3.connect(dbPath)
cursor = connection.cursor()

print(f'Connected to: {dbPath}')

Connected to: imdb_class_2010plus.db


---

## Part 1: What's in This Database?

Before we write any queries, let's see what tables we have. Remember `sqlite_master`? That's the system table that tells us what's inside the database.

In [25]:
# List all tables in the database
tableQuery = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
tables = cursor.execute(tableQuery).fetchall()

print("Tables in imdb_class_2010plus.db:")
print("-" * 35)
for i, (tableName,) in enumerate(tables, 1):
    # Also get the row count for each table
    count = cursor.execute(f"SELECT COUNT(*) FROM {tableName}").fetchone()[0]
    print(f"  {i}. {tableName:25s} ({count:,} rows)")

Tables in imdb_class_2010plus.db:
-----------------------------------
  1. name_basics               (182,015 rows)
  2. title_basics              (19,462 rows)
  3. title_crew_person         (116,970 rows)
  4. title_principals          (364,848 rows)
  5. title_ratings             (19,462 rows)


### Quick Overview

Here's what each table holds:

| Table | What it stores | Think of it as... |
|-------|---------------|-------------------|
| **title_basics** | Movie/show info (title, year, genre, runtime) | The Netflix browse page |
| **title_ratings** | IMDb ratings and vote counts | The star ratings you see on IMDb |
| **title_principals** | Who worked on what (actors, directors, etc.) | The movie credits |
| **title_crew_person** | Directors and writers for each title | The "Directed by" credit |
| **name_basics** | People's names, birth years, professions | The actor/director profiles |

Let's look at each one.

### Table 1: title_basics

This is the main table — one row per movie or TV series.

In [26]:
# Look at the columns and a few sample rows
print("=== title_basics ===\n")

# Show column info
schema = cursor.execute("PRAGMA table_info(title_basics)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

# Sample rows
df = pd.read_sql_query("SELECT * FROM title_basics LIMIT 5", connection)
print(df.to_string(index=False))

=== title_basics ===

  tconst               TEXT
  titleType            TEXT
  primaryTitle         TEXT
  originalTitle        TEXT
  isAdult              INTEGER
  startYear            INTEGER
  endYear              INTEGER
  runtimeMinutes       INTEGER
  genres               TEXT

   tconst titleType               primaryTitle              originalTitle  isAdult  startYear endYear  runtimeMinutes                     genres
tt0069049     movie The Other Side of the Wind The Other Side of the Wind        0       2018    None             122                      Drama
tt0249516     movie                 Foodfight!                 Foodfight!        0       2012    None              91 Adventure,Animation,Comedy
tt0293069     movie                 Dark Blood                 Dark Blood        0       2012    None              86                   Thriller
tt0293429     movie              Mortal Kombat              Mortal Kombat        0       2021    None             110   Action,Advent

**Key columns to know:**
- `tconst` — the unique ID for each title (like "tt1375666" for Inception). Think of this like a student ID but for movies.
- `titleType` — either "movie" or "tvSeries"
- `primaryTitle` — the title you'd recognize
- `startYear` — when it came out
- `genres` — comma-separated list like "Action,Adventure,Sci-Fi"
- `runtimeMinutes` — how long it is

### Table 2: title_ratings

Every title's IMDb rating and how many people voted.

In [27]:
# Sample from title_ratings
print("=== title_ratings ===\n")

schema = cursor.execute("PRAGMA table_info(title_ratings)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM title_ratings LIMIT 5", connection)
print(df.to_string(index=False))

=== title_ratings ===

  tconst               TEXT
  averageRating        REAL
  numVotes             INTEGER

   tconst  averageRating  numVotes
tt0069049            6.7      8435
tt0249516            1.3     12569
tt0293069            6.3      1519
tt0293429            6.1    201744
tt0323808            3.8      3145


**Key columns:**
- `tconst` — matches up with title_basics (same movie ID)
- `averageRating` — the IMDb score (1.0 to 10.0)
- `numVotes` — how many people rated it (more votes = more trustworthy rating)

### Table 3: title_principals

This connects **people** to **titles**. If Nicolas Cage is in a movie, there's a row here linking his ID to that movie's ID.

In [28]:
# Sample from title_principals
print("=== title_principals ===\n")

schema = cursor.execute("PRAGMA table_info(title_principals)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM title_principals LIMIT 5", connection)
print(df.to_string(index=False))

=== title_principals ===

  tconst               TEXT
  ordering             INTEGER
  nconst               TEXT
  category             TEXT
  job                  TEXT
  characters           TEXT

   tconst  ordering    nconst category  job           characters
tt0069049         1 nm0001379    actor None   ["Jake Hannaford"]
tt0069049         2 nm0462648  actress None      ["The Actress"]
tt0069049         3 nm0000953    actor None ["Brooks Otterlake"]
tt0069049         4 nm0001782  actress None       ["Julie Rich"]
tt0069049         5 nm0287988    actor None      ["Billy Boyle"]


**Key columns:**
- `tconst` — the movie/show ID (links to title_basics)
- `nconst` — the person ID (links to name_basics)
- `category` — what they did: "actor", "actress", "director", "writer", "producer", etc.
- `characters` — the character name(s) they played

This is the table that lets us answer questions like "What movies was Samuel L. Jackson in?" or "Who acted in Inception?"

### Table 4: name_basics

The people table — actors, directors, writers, etc.

In [29]:
# Sample from name_basics
print("=== name_basics ===\n")

schema = cursor.execute("PRAGMA table_info(name_basics)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM name_basics LIMIT 5", connection)
print(df.to_string(index=False))

=== name_basics ===

  nconst               TEXT
  primaryName          TEXT
  birthYear            INTEGER
  deathYear            INTEGER
  primaryProfession    TEXT
  knownForTitles       TEXT

   nconst       primaryName  birthYear  deathYear                    primaryProfession                          knownForTitles
nm0000148     Harrison Ford       1942        NaN                actor,writer,producer tt0082971,tt0076759,tt0106977,tt0090329
nm0000375 Robert Downey Jr.       1965        NaN                actor,producer,writer tt0371746,tt1300854,tt4154796,tt0848228
nm0000829      Stuart Baird       1947        NaN editor,editorial_department,director tt1074638,tt0078346,tt0381061,tt0253754
nm0001039     David Charvet       1972        NaN     actor,soundtrack,archive_footage tt0103491,tt0496375,tt0096542,tt0112464
nm0001358      Hal Holbrook       1925     2021.0                actor,director,writer tt0758758,tt0443272,tt0077294,tt0074119


**Key columns:**
- `nconst` — unique person ID (like "nm0000115" for Nicolas Cage)
- `primaryName` — their name
- `birthYear` / `deathYear` — when they were born (and died, if applicable)
- `primaryProfession` — what they're known for
- `knownForTitles` — comma-separated list of their most famous title IDs

### Table 5: title_crew_person

Directors and writers for each title. Similar to title_principals but specifically for crew roles.

In [30]:
# Sample from title_crew_person
print("=== title_crew_person ===\n")

schema = cursor.execute("PRAGMA table_info(title_crew_person)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

print()

df = pd.read_sql_query("SELECT * FROM title_crew_person LIMIT 5", connection)
print(df.to_string(index=False))

=== title_crew_person ===

  tconst               TEXT
  nconst               TEXT
  role                 TEXT

   tconst    nconst     role
tt0069049 nm0000080 director
tt0249516 nm0440415 director
tt0293069 nm0806293 director
tt0293429 nm2585406 director
tt0323808 nm0362736 director


---

## Part 2: SQL Refresher — The Basics

Let's warm up with queries you've seen before. Same SQL, new data.

### SELECT + WHERE

Remember: `SELECT` picks columns, `WHERE` filters rows.

Let's find some movies you know.

In [31]:
# Find Inception
query = '''
    SELECT primaryTitle, startYear, genres, runtimeMinutes
    FROM title_basics
    WHERE primaryTitle = 'Inception'
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

primaryTitle  startYear                  genres  runtimeMinutes
   Inception       2010 Action,Adventure,Sci-Fi             148


#### New SQL Concept: `LIKE` with Wildcards

Sometimes you don't know the exact title — you just want to search for a keyword. That's what `LIKE` does.

- `LIKE '%Avengers%'` — finds any title that **contains** "Avengers" anywhere
- `LIKE 'The%'` — finds titles that **start with** "The"
- `LIKE '%Man'` — finds titles that **end with** "Man"

The `%` is a **wildcard** — it means "any number of characters." Think of it like the `*` in a Google search.

**Important:** `LIKE` is **case-sensitive** in SQLite by default for ASCII characters. So `'spider'` won't match `'Spider'`.

In [32]:
# Find all Marvel movies (titles containing "Avengers")
query = '''
    SELECT primaryTitle, startYear, genres
    FROM title_basics
    WHERE primaryTitle LIKE '%Avengers%'
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

                          primaryTitle  startYear                     genres
The Avengers: Earth's Mightiest Heroes       2010 Action,Adventure,Animation
                          The Avengers       2012              Action,Sci-Fi
                     Avengers Assemble       2012 Action,Adventure,Animation
                            Scavengers       2013     Action,Sci-Fi,Thriller
               Avengers: Age of Ultron       2015    Action,Adventure,Sci-Fi
                Avengers: Infinity War       2018    Action,Adventure,Sci-Fi
                     Avengers: Endgame       2019    Action,Adventure,Sci-Fi


### Try It: Find Your Favorites

Change the query below to search for a movie or show YOU like. Try:
- An exact title: `WHERE primaryTitle = 'Your Movie'`
- A partial match: `WHERE primaryTitle LIKE '%keyword%'`

In [33]:
# YOUR TURN: Search for a movie or show you like
query = '''
    SELECT primaryTitle, startYear, genres, runtimeMinutes
    FROM title_basics
    WHERE primaryTitle LIKE '%Spider%'
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

                         primaryTitle  startYear                     genres  runtimeMinutes
                           Spiderhole       2010                     Horror            82.0
               The Amazing Spider-Man       2012              Action,Sci-Fi           136.0
                  Ultimate Spider-Man       2012 Action,Adventure,Animation            23.0
                      Big Ass Spider!       2013    Action,Adventure,Comedy            80.0
             The Amazing Spider-Man 2       2014    Action,Adventure,Sci-Fi           142.0
               Spider-Man: Homecoming       2017    Action,Adventure,Sci-Fi           133.0
                           Spider-Man       2017 Action,Adventure,Animation            44.0
    Spider-Man: Into the Spider-Verse       2018 Action,Adventure,Animation           117.0
         The Girl in the Spider's Web       2018     Action,Adventure,Crime           115.0
            Spider-Man: Far from Home       2019    Action,Adventure,Comedy     

### ORDER BY + LIMIT

Remember: `ORDER BY` sorts results, `LIMIT` caps how many rows you get back.

#### New SQL Concepts: `JOIN`, `ON`, and Table Aliases

The next query pulls data from **two tables at once**. To do that, we need a few new tools:

**`JOIN ... ON`** — Combines rows from two tables where a column matches.
```sql
FROM title_basics b
JOIN title_ratings r ON b.tconst = r.tconst
```
This says: "Take each row from `title_basics`, find the matching row in `title_ratings` where the `tconst` values are the same, and glue them together."

**Table Aliases** — The `b` and `r` are shortcuts (aliases) so we don't have to type the full table name every time:
- `b` = `title_basics`
- `r` = `title_ratings`

Then we use `b.primaryTitle` instead of `title_basics.primaryTitle`. Same thing, less typing.

**Why do we need aliases?** Both tables have a column called `tconst`. If we just write `tconst`, SQL doesn't know which table we mean. The alias makes it clear: `b.tconst` vs `r.tconst`.

In [34]:
# Top 10 highest-rated movies with at least 100,000 votes
# (We join title_basics and title_ratings to get both title info AND ratings)
query = '''
    SELECT b.primaryTitle, b.startYear, r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND r.numVotes > 100000
    ORDER BY r.averageRating DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print("Top 10 highest-rated movies (100K+ votes):")
print(df.to_string(index=False))

Top 10 highest-rated movies (100K+ votes):
                       primaryTitle  startYear  averageRating  numVotes
                          Inception       2010            8.8   2767665
                       Interstellar       2014            8.7   2453389
                   Django Unchained       2012            8.5   1850242
                           Whiplash       2014            8.5   1124940
Spider-Man: Across the Spider-Verse       2023            8.5    499492
              The Dark Knight Rises       2012            8.4   1964816
                     Dune: Part Two       2024            8.4    703512
                               Coco       2017            8.4    687947
             Avengers: Infinity War       2018            8.4   1341613
                  Avengers: Endgame       2019            8.4   1413670


#### New SQL Concept: `IS NOT NULL`

Some movies are missing their runtime — the value is `NULL` (blank/unknown). If we're sorting by runtime, we want to skip those.

```sql
WHERE runtimeMinutes IS NOT NULL
```

This filters out any row where `runtimeMinutes` has no value. You can't use `= NULL` or `!= NULL` in SQL — it **must** be `IS NULL` or `IS NOT NULL`. That's just how SQL works.

In [35]:
# Longest movies in the database
query = '''
    SELECT primaryTitle, startYear, runtimeMinutes, genres
    FROM title_basics
    WHERE runtimeMinutes IS NOT NULL
      AND titleType = 'movie'
    ORDER BY runtimeMinutes DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print("Longest movies:")
print(df.to_string(index=False))

Longest movies:
                                                         primaryTitle  startYear  runtimeMinutes                        genres
                                              Europa: The Last Battle       2017             746       Documentary,History,War
                                                         Paint Drying       2016             607                   Documentary
                                                O.J.: Made in America       2016             467   Biography,Crime,Documentary
                          Adolf Hitler: The Greatest Story Never Told       2013             350 Biography,Documentary,History
   In Search of Darkness Part III: The Final Journey Into '80s Horror       2022             341    Documentary,History,Horror
                                                In Search of Tomorrow       2022             305    Documentary,History,Sci-Fi
                                                        Occupied City       2023             26

### Try It: Your Own Ranking

Modify the query below to find YOUR top 10. Some ideas:
- Most popular movies from a specific year
- Highest rated TV series
- Shortest movies
- Movies from a specific genre (hint: `WHERE genres LIKE '%Horror%'`)

### Best genres to try first
- Action
- Comedy
- Drama
- Horror
- Thriller
- Sci-Fi
- Adventure
- Animation
- Crime

### Best genre combinations to try
- Action,Adventure
- Action,Comedy
- Action,Sci-Fi
- Action,Thriller
- Adventure,Fantasy
- Adventure,Sci-Fi
- Comedy,Drama
- Comedy,Romance
- Crime,Drama
- Horror,Thriller
- Mystery,Thriller

In [43]:
# YOUR TURN: Create your own top 10 list
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, r.averageRating, r.numVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND b.genres LIKE '%Adventure%'
      AND r.numVotes > 10000
    ORDER BY r.averageRating DESC
    LIMIT 10
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

                       primaryTitle  startYear                     genres  averageRating  numVotes
                          Inception       2010    Action,Adventure,Sci-Fi            8.8   2767665
                       Interstellar       2014     Adventure,Drama,Sci-Fi            8.7   2453389
         Solo Leveling: ReAwakening       2024 Action,Adventure,Animation            8.7     15989
Spider-Man: Across the Spider-Verse       2023 Action,Adventure,Animation            8.5    499492
                     Dune: Part Two       2024     Action,Adventure,Drama            8.4    703512
                               Coco       2017  Adventure,Animation,Drama            8.4    687947
             Avengers: Infinity War       2018    Action,Adventure,Sci-Fi            8.4   1341613
                  Avengers: Endgame       2019    Action,Adventure,Sci-Fi            8.4   1413670
  Spider-Man: Into the Spider-Verse       2018 Action,Adventure,Animation            8.4    768030
          

---

### GROUP BY + Aggregations

Remember: `GROUP BY` groups rows together so you can use `COUNT()`, `AVG()`, `MAX()`, `MIN()`, `SUM()` on each group.

#### New SQL Concepts: `ROUND()` and `AVG()`

You already know `COUNT(*)` counts rows. Two more aggregation functions:

**`AVG(column)`** — Calculates the average (mean) of a numeric column.
```sql
AVG(r.averageRating)   -- average of all ratings in the group
```

**`ROUND(value, decimals)`** — Rounds a number to a specified number of decimal places.
```sql
ROUND(AVG(r.averageRating), 2)   -- average rounded to 2 decimal places
ROUND(AVG(r.numVotes), 0)        -- average rounded to 0 decimals (whole number)
```

Without `ROUND()`, you'd get numbers like `5.873291046...` which are hard to read. `ROUND(..., 2)` gives you `5.87`.

In [44]:
# How many movies came out each year?
query = '''
    SELECT startYear, COUNT(*) as movieCount
    FROM title_basics
    WHERE titleType = 'movie'
      AND startYear >= 2015
    GROUP BY startYear
    ORDER BY startYear
'''

df = pd.read_sql_query(query, connection)
print("Movies per year (2015+):")
print(df.to_string(index=False))

Movies per year (2015+):
 startYear  movieCount
      2015         943
      2016         999
      2017        1047
      2018        1053
      2019        1051
      2020         859
      2021         912
      2022        1105
      2023        1071
      2024         955
      2025         653


In [45]:
# Average rating by year — are movies getting better or worse?
query = '''
    SELECT b.startYear, 
           COUNT(*) as movieCount,
           ROUND(AVG(r.averageRating), 2) as avgRating,
           ROUND(AVG(r.numVotes), 0) as avgVotes
    FROM title_basics b
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE b.titleType = 'movie'
      AND b.startYear >= 2015
    GROUP BY b.startYear
    ORDER BY b.startYear
'''

df = pd.read_sql_query(query, connection)
print("Movie ratings by year:")
print(df.to_string(index=False))

Movie ratings by year:
 startYear  movieCount  avgRating  avgVotes
      2015         943       5.87   34835.0
      2016         999       5.89   35048.0
      2017        1047       5.87   32542.0
      2018        1053       5.87   29014.0
      2019        1051       5.83   29649.0
      2020         859       5.73   16887.0
      2021         912       5.84   24111.0
      2022        1105       5.90   21623.0
      2023        1071       6.02   19484.0
      2024         955       6.18   17415.0
      2025         653       6.37   16592.0


In [46]:
# Which categories of people work on movies? (actor, director, writer, etc.)
query = '''
    SELECT category, COUNT(*) as totalCredits
    FROM title_principals
    GROUP BY category
    ORDER BY totalCredits DESC
'''

df = pd.read_sql_query(query, connection)
print("Credits by category:")
print(df.to_string(index=False))

Credits by category:
           category  totalCredits
              actor        112938
            actress         69835
           producer         37463
             writer         31180
               self         19218
             editor         18329
           composer         16528
           director         16154
    cinematographer         15713
   casting_director         14277
production_designer         11417
    archive_footage          1775
      archive_sound            21


### Try It: Your Own GROUP BY

Ideas to try:
- Count movies by genre (tricky — genres are comma-separated, so `LIKE '%Action%'` with GROUP BY year)
- Find the year with the most TV series
- Average runtime by titleType (movie vs tvSeries)

In [47]:
# YOUR TURN: Write a GROUP BY query
query = '''
    SELECT titleType, 
           COUNT(*) as count,
           ROUND(AVG(runtimeMinutes), 1) as avgRuntime
    FROM title_basics
    WHERE runtimeMinutes IS NOT NULL
    GROUP BY titleType
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

titleType  count  avgRuntime
    movie  14654       103.0
 tvSeries   4012        45.1


---

## Part 3: Connecting Tables (JOINs)

The real power of this database comes when we connect tables together. We've been doing this already with `JOIN` — let's make it explicit.

#### New SQL Concepts: `IN ()` and `DISTINCT`

Two more tools you'll see in the JOIN queries below:

**`IN (value1, value2, ...)`** — A shortcut for multiple `OR` conditions.
```sql
WHERE p.category IN ('actor', 'actress')
```
This is the same as writing:
```sql
WHERE p.category = 'actor' OR p.category = 'actress'
```
Much cleaner when you have several values to check!

**`DISTINCT`** — Removes duplicate values from a result.
```sql
COUNT(DISTINCT p.tconst)   -- counts unique movie IDs only
```
Without `DISTINCT`, if an actor has two credits on the same movie (like appearing in two roles), it would count that movie twice. `DISTINCT` makes sure each movie is counted only once.

In [48]:
# Who are the most prolific actors in this database?
query = '''
    SELECT n.primaryName, COUNT(DISTINCT p.tconst) as movieCount
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    WHERE p.category IN ('actor', 'actress')
    GROUP BY n.nconst
    ORDER BY movieCount DESC
    LIMIT 15
'''

df = pd.read_sql_query(query, connection)
print("Most prolific actors (by number of titles):")
print(df.to_string(index=False))

Most prolific actors (by number of titles):
      primaryName  movieCount
     J.K. Simmons          53
     Nicolas Cage          52
     Eric Roberts          52
     Bruce Willis          50
      Danny Trejo          50
     Frank Grillo          46
  Fred Tatasciore          46
Samuel L. Jackson          45
  Dermot Mulroney          45
     Grey DeLisle          44
  Michael Shannon          44
       Judy Greer          42
     Willem Dafoe          41
   Dolph Lundgren          39
  Woody Harrelson          39


In [49]:
# What movies has a specific actor been in?
# Let's try Nicolas Cage
query = '''
    SELECT b.primaryTitle, b.startYear, b.genres, r.averageRating
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE n.primaryName = 'Nicolas Cage'
      AND p.category = 'actor'
      AND b.titleType = 'movie'
    ORDER BY b.startYear DESC
'''

df = pd.read_sql_query(query, connection)
print(f"Nicolas Cage movies ({len(df)} total):")
print(df.to_string(index=False))

Nicolas Cage movies (55 total):
                           primaryTitle  startYear                     genres  averageRating
                            Gunslingers       2025       Action,Drama,Western            3.6
                    The Carpenter's Son       2025                     Horror            4.3
                               Arcadian       2024        Action,Drama,Horror            5.5
                               Longlegs       2024       Crime,Horror,Mystery            6.6
                             The Surfer       2024                   Thriller            5.9
                               Renfield       2023      Action,Comedy,Fantasy            6.4
                    The Retirement Plan       2023        Action,Comedy,Crime            5.1
                         Dream Scenario       2023       Comedy,Drama,Fantasy            6.8
                 Sympathy for the Devil       2023            Action,Thriller            5.5
                            The Old Wa

### Try It: Look Up YOUR Favorite Actor

Change the name in the query below to find movies for an actor you like.

In [50]:
# YOUR TURN: Pick an actor and find their movies
query = '''
    SELECT b.primaryTitle, b.startYear, r.averageRating, p.characters
    FROM name_basics n
    JOIN title_principals p ON n.nconst = p.nconst
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN title_ratings r ON b.tconst = r.tconst
    WHERE n.primaryName = 'Zendaya'
      AND p.category IN ('actor', 'actress')
    ORDER BY b.startYear DESC
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

             primaryTitle  startYear  averageRating          characters
           Dune: Part Two       2024            8.4           ["Chani"]
              Challengers       2024            7.0 ["Tashi Donaldson"]
  Spider-Man: No Way Home       2021            8.2              ["MJ"]
           Dune: Part One       2021            8.0           ["Chani"]
          Malcolm & Marie       2021            6.6           ["Marie"]
Spider-Man: Far from Home       2019            7.3              ["MJ"]
                 Euphoria       2019            8.2     ["Rue Bennett"]
          Duck Duck Goose       2018            5.8             ["Chi"]
                Smallfoot       2018            6.6         ["Meechee"]
     The Greatest Showman       2017            7.5    ["Anne Wheeler"]
   Spider-Man: Homecoming       2017            7.4        ["Michelle"]
          K.C. Undercover       2015            6.5     ["K.C. Cooper"]
              Shake It Up       2010            5.0      ["Rocky

In [51]:
# Who starred in a specific movie?
query = '''
    SELECT n.primaryName, p.category, p.characters
    FROM title_basics b
    JOIN title_principals p ON b.tconst = p.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE b.primaryTitle = 'Inception'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''

df = pd.read_sql_query(query, connection)
print("Cast & director of Inception:")
print(df.to_string(index=False))

Cast & director of Inception:
         primaryName category              characters
   Leonardo DiCaprio    actor                ["Cobb"]
Joseph Gordon-Levitt    actor              ["Arthur"]
         Elliot Page    actor             ["Ariadne"]
        Ken Watanabe    actor               ["Saito"]
           Tom Hardy    actor               ["Eames"]
          Dileep Rao    actor               ["Yusuf"]
      Cillian Murphy    actor ["Robert Fischer, Jr."]
        Tom Berenger    actor            ["Browning"]
    Marion Cotillard  actress                 ["Mal"]
  Pete Postlethwaite    actor     ["Maurice Fischer"]
   Christopher Nolan director                    None


### Try It: Who's in YOUR Favorite Movie?

Change the title below to look up the cast of a movie you love.

In [52]:
# YOUR TURN: Look up the cast of a movie
query = '''
    SELECT n.primaryName, p.category, p.characters
    FROM title_basics b
    JOIN title_principals p ON b.tconst = p.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE b.primaryTitle = 'The Batman'
      AND p.category IN ('actor', 'actress', 'director')
    ORDER BY p.ordering
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

     primaryName category                       characters
Robert Pattinson    actor                  ["Bruce Wayne"]
Robert Pattinson    actor                   ["The Batman"]
     Zoë Kravitz  actress                  ["Selina Kyle"]
  Jeffrey Wright    actor             ["Lt. James Gordon"]
   Colin Farrell    actor                           ["Oz"]
   Colin Farrell    actor                  ["The Penguin"]
       Paul Dano    actor                  ["The Riddler"]
   John Turturro    actor              ["Carmine Falcone"]
     Andy Serkis    actor                       ["Alfred"]
 Peter Sarsgaard    actor ["District Attorney Gil Colson"]
   Barry Keoghan    actor       ["Unseen Arkham Prisoner"]
    Jayme Lawson  actress                   ["Bella Reál"]
     Matt Reeves director                             None


---

## Wrap-Up

**Today we refreshed:**
- `SELECT` / `WHERE` / `ORDER BY` / `LIMIT` — picking and filtering data
- `GROUP BY` with `COUNT()`, `AVG()` — summarizing data
- `JOIN` — connecting tables together

**The IMDb database has 5 tables:**
- `title_basics` and `title_ratings` hold movie/show info
- `name_basics` holds people info
- `title_principals` and `title_crew_person` connect people to titles

**Next up:** Your task notebook — explore this database on your own!

In [53]:
# Clean up
connection.close()
print("Database connection closed.")

Database connection closed.
